<a href="https://colab.research.google.com/github/gautamkr1876/AIML_ClassNotes/blob/main/8.%20Agentic%20AI%20Systems/14.%20DSPy%3A%20Mathematical%20Prompt%20Optimization/DSPY_liveclass.ipynb" target="_blank">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

## Environment Setup

In [ ]:
!pip install -q -U "dspy>=2.5" datasets langchain-core groq matplotlib


import dspy
print("Install complete. dspy version:", dspy.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 7.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.2/421.2 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.2/290.2 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.0/572.0 kB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.3/51.3 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.4/27.4 MB 78.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os
import random
import time
import getpass
from collections import Counter

import dspy
from datasets import load_dataset
import matplotlib.pyplot as plt

# Reproducibility
random.seed(42)

print("DSPy version:", dspy.__version__)

DSPy version: 3.3.1


In [ ]:
# Groq API key handling. Two paths:
#   (a) Colab: store as a secret named GROQ_API_KEY, we'll auto-pick it up
#   (b) Otherwise: paste when prompted

api_key = None
try:
    from google.colab import userdata
    api_key = userdata.get('GROQ_API_KEY')
    print("Loaded key from Colab secrets.")
except Exception:
    pass

if not api_key:
    api_key = getpass.getpass("Paste your Groq API key: ").strip()

os.environ["GROQ_API_KEY"] = api_key
assert api_key.startswith("gsk_"), "Groq keys start with 'gsk_'. Please recheck."
print("API key registered.")

Paste your Groq API key: ··········
API key registered.


In [ ]:
lm = dspy.LM(
    model="groq/openai/gpt-oss-20b",
    api_key=os.environ["GROQ_API_KEY"],
    max_tokens=512,
    temperature=0.0,   # deterministic-ish for classification
)
dspy.configure(lm=lm)

# Sanity check: one round-trip
resp = lm("Reply with exactly the word: pong")
print("Sanity check response:", resp)

Sanity check response: ['pong']


## Quiz

**Question:** You want your LLM to classify legal contracts by risk level, AND you want to see its reasoning before the final label so you can audit it. Which combination is correct?

**A.** Write one Signature with a longer prompt string

**B.** Write two separate Signatures, one for reasoning and one for classification

**C.** Write one Signature (`contract -> risk_level`) and wrap it in `dspy.ChainOfThought`

**D.** Write one Signature (`contract -> risk_level`) and wrap it in `dspy.Predict`, then post-process


**Correct: C**

## About the data


We'll use the **Financial PhraseBank** dataset (Malo et al., 2014), specifically the `sentences_75agree` split where at least 75% of human annotators agreed on the label. This gives us ~3400 high-quality labeled headlines.

Labels: `positive`, `negative`, `neutral`.

### Why financial sentiment is genuinely hard

Consider these three headlines:

1. *"Fed signals patience on rate cuts"*
   - Sounds **neutral**. But for growth stocks: **negative** (higher rates for longer).

2. *"Company X beats Q3 earnings by $0.02"*
   - Sounds **positive**. But if analysts expected $0.10 beat: market reads as **negative**.

3. *"Regulator opens preliminary inquiry into X"*
   - Sounds **negative**. But often priced in as **neutral** (inquiries rarely lead to action).

Generic sentiment classifiers get roasted here because they miss **domain semantics**. This is exactly the gap DSPy is going to close for us, without us hand-writing any of these rules.

In [ ]:
# Load Financial PhraseBank
# Note: this dataset requires `trust_remote_code=True` on some versions

raw = load_dataset("lmassaron/FinancialPhraseBank", trust_remote_code=True)
print(raw)
print("First 3 examples:")
for i in range(3):
    print(raw['train'][i])

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'lmassaron/FinancialPhraseBank' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'lmassaron/FinancialPhraseBank' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


README.md:   0%|          | 0.00/2.44k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  332kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 43.2kB            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/3872 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/484 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/484 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentiment', 'sentence', 'label'],
        num_rows: 3872
    })
    validation: Dataset({
        features: ['sentiment', 'sentence', 'label'],
        num_rows: 484
    })
    test: Dataset({
        features: ['sentiment', 'sentence', 'label'],
        num_rows: 484
    })
})
First 3 examples:
{'sentiment': 'neutral', 'sentence': 'The shares carry a right to dividend and other shareholder rights as from their registration with the Finnish Trade Register .', 'label': 1}
{'sentiment': 'positive', 'sentence': 'UPM-Kymmene has generated four consecutive quarters of positive Free Cash Flow .', 'label': 2}
{'sentiment': 'positive', 'sentence': 'Known as Post Bank , the concept would see Fidelity Bank rolling out 75 offices in Ghana Post premises , to provide financial services to the people .', 'label': 2}


In [ ]:

for i in range(10):
    print(raw['train'][i])

{'sentiment': 'neutral', 'sentence': 'The shares carry a right to dividend and other shareholder rights as from their registration with the Finnish Trade Register .', 'label': 1}
{'sentiment': 'positive', 'sentence': 'UPM-Kymmene has generated four consecutive quarters of positive Free Cash Flow .', 'label': 2}
{'sentiment': 'positive', 'sentence': 'Known as Post Bank , the concept would see Fidelity Bank rolling out 75 offices in Ghana Post premises , to provide financial services to the people .', 'label': 2}
{'sentiment': 'negative', 'sentence': "Hobby Hall 's sales decrease 26 pct due to implementing a new information system that involved changing in the principal of posting sales .", 'label': 0}
{'sentiment': 'neutral', 'sentence': 'The total restructuring costs are expected to be about EUR 30mn , of which EUR 13.5 mn was booked in December 2008 .', 'label': 1}
{'sentiment': 'negative', 'sentence': 'The company confirmed its estimate for lower revenue for the whole 2009 than the y

In [ ]:
# Map integer labels to strings
label_map = {0: "negative", 1: "neutral", 2: "positive"}

all_examples = []
for row in raw['train']:
    all_examples.append({
        "headline": row['sentence'],
        "sentiment": label_map[row['label']]
    })

print(f"Total examples: {len(all_examples)}")
print("Class distribution:", Counter(ex['sentiment'] for ex in all_examples))


Total examples: 3872
Class distribution: Counter({'neutral': 2298, 'positive': 1091, 'negative': 483})


In [ ]:
# Stratified sampling: we want balanced train/val/test to avoid the model
# just learning "predict neutral, get 60%"

def stratified_sample(examples, per_class):
    by_class = {"positive": [], "negative": [], "neutral": []}
    for ex in examples:
        by_class[ex['sentiment']].append(ex)
    for k in by_class:
        random.shuffle(by_class[k])
    sampled = []
    for k in by_class:
        sampled.extend(by_class[k][:per_class])
    random.shuffle(sampled)
    return sampled, {k: by_class[k][per_class:] for k in by_class}

# Shuffle the master pool first
random.shuffle(all_examples)

# Split: 20 train (for teleprompter), 50 val (for optimizer scoring), 100 test (final eval)
# We take a "per class" count so we're balanced.
train_raw, remaining = stratified_sample(all_examples, per_class=7)   # 21 total (7*3)
# Reflatten remaining pool
remaining_flat = sum(remaining.values(), [])
random.shuffle(remaining_flat)

val_raw, _ = stratified_sample(remaining_flat, per_class=17)  # 51 total
# Reflatten leftover
leftover = [ex for ex in remaining_flat if ex not in val_raw]

test_raw, _ = stratified_sample(leftover, per_class=34)  # 102 total

print(f"Train: {len(train_raw)} | Val: {len(val_raw)} | Test: {len(test_raw)}")
print("Train distribution:", Counter(e['sentiment'] for e in train_raw))
print("Val distribution:  ", Counter(e['sentiment'] for e in val_raw))
print("Test distribution: ", Counter(e['sentiment'] for e in test_raw))


Train: 21 | Val: 51 | Test: 102
Train distribution: Counter({'negative': 7, 'neutral': 7, 'positive': 7})
Val distribution:   Counter({'negative': 17, 'neutral': 17, 'positive': 17})
Test distribution:  Counter({'neutral': 34, 'positive': 34, 'negative': 34})


In [ ]:
# Convert to DSPy Examples. This is the format teleprompters expect.
def to_dspy(examples):
    return [
        dspy.Example(headline=e['headline'], sentiment=e['sentiment']).with_inputs("headline")
        for e in examples
    ]

trainset = to_dspy(train_raw)
valset = to_dspy(val_raw)
testset = to_dspy(test_raw)

print("Example DSPy record:")
print(trainset[0])
print("\nInputs:", trainset[0].inputs())
print("Labels:", trainset[0].labels())


Example DSPy record:
Example({'headline': "The fair value of the company 's investment properties went down to EUR 2.768 billion at the end of 2009 from EUR 2.916 billion a year earlier .", 'sentiment': 'negative'}) (input_keys={'headline'})

Inputs: Example({'headline': "The fair value of the company 's investment properties went down to EUR 2.768 billion at the end of 2009 from EUR 2.916 billion a year earlier ."}) (input_keys={'headline'})
Labels: Example({'sentiment': 'negative'}) (input_keys=None)


In [ ]:
for i in range(5):
    ex = testset[i]
    print(f"[{ex.sentiment.upper():8}] {ex.headline}")
    print()

[NEUTRAL ] Panostaja Oyj 's Board also decided at its organisational meeting held upon completion of the AGM to implement the AGM decision concerning Board member fees paid as shares in such a way that shares are transferred on a quarterly basis on the date following publication of the quarterly-annual report .

[POSITIVE] `` They would invest not only in the physical infrastructure , but would also provide know-how for managing and developing science and technology parks , '' said Sunrise Valley director Andrius Bagdonas .

[POSITIVE] With the acquisition , the company will expand its offering to North , Central and South America , it said .

[NEUTRAL ] The Board of Directors proposes to the Shareholders ' Meeting on 18 March 2010 that the company would pay dividend for the financial year January 1 - December 31 , 2009 , EUR 0.02 per share .

[NEUTRAL ] The Finland-based company says it will move into an existing 260,000-square-foot facility in September .



## DSPy Signatures & Modules

In [ ]:
## Version 1: The naive signature

class FinancialSentimentBasic(dspy.Signature):
    """Classify the sentiment of a financial news headline."""
    headline: str = dspy.InputField()
    sentiment: str = dspy.OutputField(desc="one of: positive, negative, neutral")

# Wrap in Predict (single-shot, no reasoning)
basic_classifier = dspy.Predict(FinancialSentimentBasic) # Module

# Try it
sample = "Nokia's third-quarter profit fell short of analyst expectations"
result = basic_classifier(headline=sample)
print("Headline:", sample)
print("Predicted:", result.sentiment)

Headline: Nokia's third-quarter profit fell short of analyst expectations
Predicted: negative


Notice: **you did not write that prompt**. DSPy generated it from the signature. This is the "declarative" part of DSPy.

In [ ]:
# Peek at the last prompt DSPy sent
dspy.inspect_history(n=1)





[2026-08-13T17:29:14.535246]

System message:

Your input fields are:
1. `headline` (str):
Your output fields are:
1. `sentiment` (str): one of: positive, negative, neutral
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## headline ## ]]
{headline}

[[ ## sentiment ## ]]
{sentiment}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Classify the sentiment of a financial news headline.


User message:

[[ ## headline ## ]]
Nokia's third-quarter profit fell short of analyst expectations

Respond with the corresponding output fields, starting with the field `[[ ## sentiment ## ]]`, and then ending with the marker for `[[ ## completed ## ]]`.


Response:

[[ ## sentiment ## ]]
negative

[[ ## completed ## ]]







In [ ]:
## Version 2: Adding Chain of Thought
## Same signature, different module.

cot_classifier = dspy.ChainOfThought(FinancialSentimentBasic) # Modules/ Strategy

result = cot_classifier(headline=sample)
print("Reasoning:", result.reasoning)
print("Sentiment:", result.sentiment)


Reasoning: The headline indicates that Nokia's third‑quarter profit did not meet analyst expectations, implying a shortfall in performance. This is a negative development for the company and its investors, suggesting a downturn or weaker financial results than anticipated.
Sentiment: negative


In [ ]:
# Inspect the prompt for ChainOfThought
dspy.inspect_history(n=1)





[2026-09-24T17:13:20.931910]

System message:

Your input fields are:
1. `headline` (str):
Your output fields are:
1. `reasoning` (str): 
2. `sentiment` (str): one of: positive, negative, neutral
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## headline ## ]]
{headline}

[[ ## reasoning ## ]]
{reasoning}

[[ ## sentiment ## ]]
{sentiment}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Classify the sentiment of a financial news headline.


User message:

[[ ## headline ## ]]
Nokia's third-quarter profit fell short of analyst expectations

Respond with the corresponding output fields, starting with the field `[[ ## reasoning ## ]]`, then `[[ ## sentiment ## ]]`, and then ending with the marker for `[[ ## completed ## ]]`.


Response:

[[ ## reasoning ## ]]
The headline indicates that Nokia's third‑quarter profit did not meet analyst expectations, implying a shortfall in performance. This is a ne

In [ ]:
## Version 3: A richer signature with descriptions


class FinancialSentiment(dspy.Signature):
    """Classify the sentiment of a financial news headline from the perspective of an equity investor.

    Consider whether the news would likely cause the stock price of the mentioned entity
    to rise (positive), fall (negative), or remain roughly unchanged (neutral).
    Focus on financial implications, not general emotional tone.
    """
    headline: str = dspy.InputField(desc="a short financial news headline")
    sentiment: str = dspy.OutputField(
        desc="exactly one word: positive, negative, or neutral"
    )

richer_classifier = dspy.ChainOfThought(FinancialSentiment)
result = richer_classifier(headline=sample)
print("Reasoning:", result.reasoning)
print("Sentiment:", result.sentiment)

Reasoning: The news indicates that Nokia's third-quarter profit did not meet the expectations of analysts, which could lead to a decrease in investor confidence and potentially negatively impact the stock price. This is because a lower-than-expected profit may indicate that the company is not performing as well as anticipated, which could lead to a decrease in the stock price.
Sentiment: negative


This is still manual prompt engineering, just at a higher level of abstraction. We wrote a good docstring, but we haven't done any optimization yet.

## Baseline Evaluation

You cannot optimize what you cannot measure.

In [ ]:
## The metric function
## DSPy metrics take (example, prediction, trace=None) and return a score. For classification, exact match is fine.

In [ ]:
def sentiment_match(example, pred, trace=None):
    """Return 1.0 if predicted sentiment matches gold, else 0.0.
    Robust to case and whitespace."""
    gold = example.sentiment.strip().lower()
    predicted = (pred.sentiment or "").strip().lower()
    # Handle model verbosity like 'Positive.' or 'the sentiment is negative'
    for label in ["positive", "negative", "neutral"]:
        if label in predicted:
            predicted = label
            break
    return float(gold == predicted)

In [ ]:
# Quick test
fake_ex = dspy.Example(sentiment="positive")
fake_pred = dspy.Prediction(sentiment="Positive.")
print("Should be 1.0:", sentiment_match(fake_ex, fake_pred))

Should be 1.0: 1.0


In [ ]:
fake_pred_bad = dspy.Prediction(sentiment="negative")
print("Should be 0.0:", sentiment_match(fake_ex, fake_pred_bad))

Should be 0.0: 0.0


In [ ]:
from dspy.evaluate import Evaluate

# Groq free tier: ~30 req/min. We use num_threads=4 to stay safely under limits.
evaluator = Evaluate(
    devset=testset,
    metric=sentiment_match,
    num_threads=1,
    display_progress=True,
    display_table=0,
)

print("Evaluator ready. Test set size:", len(testset))

Evaluator ready. Test set size: 102


In [ ]:
## Baseline 1: Predict (no reasoning, no examples)

from dspy.evaluate import Evaluate

# Re-configure the evaluator with lower num_threads to avoid rate limits
# The previous evaluator (from cell dbe30f9a) had num_threads=4, which likely caused the rate limit issue.
# Setting num_threads=1 here ensures that requests are sent sequentially, respecting Groq's free tier limits.

evaluator = Evaluate(
    devset=testset,
    metric=sentiment_match,
    num_threads=1, # Reduced concurrency to avoid rate limits
    display_progress=True,
    display_table=0,
)

baseline_predict = dspy.Predict(FinancialSentiment)

print("Running baseline evaluation... (~1-2 min on Groq free tier)")
t0 = time.time()
baseline_predict_score = evaluator(baseline_predict)
# Access the numerical score from the EvaluationResult object
print(f"Baseline Predict accuracy: {baseline_predict_score.score:.2f}%")
print(f"Time: {time.time() - t0:.1f}s")

Running baseline evaluation... (~1-2 min on Groq free tier)
Average Metric: 81.00 / 102 (79.4%): 100%|██████████| 102/102 [00:06<00:00, 16.20it/s]

2026/08/13 17:29:21 INFO dspy.evaluate.evaluate: Average Metric: 81.0 / 102 (79.4%)



Baseline Predict accuracy: 79.41%
Time: 6.3s


In [ ]:
## Baseline 2: ChainOfThought (reasoning, no examples)


baseline_cot = dspy.ChainOfThought(FinancialSentiment)

print("Running CoT baseline evaluation...")
t0 = time.time()
baseline_cot_score = evaluator(baseline_cot)
# Access the numerical score from the EvaluationResult object
print(f"Baseline ChainOfThought accuracy: {baseline_cot_score.score:.2f}%")
print(f"Time: {time.time() - t0:.1f}s")

Running CoT baseline evaluation...
Average Metric: 71.00 / 96 (74.0%):  94%|█████████▍| 96/102 [00:52<00:21,  3.56s/it]

2026/08/13 17:30:17 ERROR dspy.utils.parallelizer: Error for Example({'headline': 'Both operating profit and turnover for the nine-month period increased , respectively from EUR2 .4 m and EUR43 .8 m , as compared to the corresponding period a year ago .', 'sentiment': 'positive'}) (input_keys={'headline'}): [llama-3.1-8b-instant] litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01kxgmg0n2exy9y4k8azwc3b28` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 5571, Requested 805. Please try again in 3.76s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}
. Set `provide_traceback=True` for traceback.


Average Metric: 71.00 / 96 (74.0%):  95%|█████████▌| 97/102 [00:55<00:17,  3.50s/it]

2026/08/13 17:30:20 ERROR dspy.utils.parallelizer: Error for Example({'headline': "Export accounts for about one tenth of the company 's annual turnover of one billion kroons .", 'sentiment': 'neutral'}) (input_keys={'headline'}): [llama-3.1-8b-instant] litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01kxgmg0n2exy9y4k8azwc3b28` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 5238, Requested 788. Please try again in 260ms. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}
. Set `provide_traceback=True` for traceback.


Average Metric: 72.00 / 97 (74.2%):  97%|█████████▋| 99/102 [01:00<00:08,  2.84s/it]

...

Average Metric: 73.00 / 98 (74.5%):  99%|█████████▉| 101/102 [01:05<00:02,  2.57s/it]

2026/08/13 17:30:30 ERROR dspy.utils.parallelizer: Error for Example({'headline': "According to Finnish petrol station chain St1 's managing director Kim Wiio , the company was forced to make purchases with rising prices in the first half of 2008 , and now consumer prices are going down almost daily due to competition .", 'sentiment': 'negative'}) (input_keys={'headline'}): [llama-3.1-8b-instant] litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01kxgmg0n2exy9y4k8azwc3b28` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 5576, Requested 817. Please try again in 3.93s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}
. Set `provide_traceback=True` for traceback.


Average Metric: 73.00 / 98 (74.5%): 100%|██████████| 102/102 [01:08<00:00,  1.48it/s]

2026/08/13 17:30:30 INFO dspy.evaluate.evaluate: Average Metric: 73.0 / 102 (71.6%)



Baseline ChainOfThought accuracy: 71.57%
Time: 69.0s


In [ ]:
# Collect predictions and errors
errors = []
for ex in testset[:30]:  # sample 30 for speed
    try:
        pred = baseline_cot(headline=ex.headline)
        if sentiment_match(ex, pred) < 1.0:
            errors.append({
                "headline": ex.headline,
                "gold": ex.sentiment,
                "predicted": pred.sentiment,
                "reasoning": pred.reasoning[:200] if hasattr(pred, "reasoning") else "n/a"
            })
    except Exception as e:
        print("Error on:", ex.headline[:60], "->", e)

print(f"Errors in first 30: {len(errors)}\n")
for i, e in enumerate(errors[:5], 1):
    print(f"--- Error {i} ---")
    print(f"Headline: {e['headline']}")
    print(f"Gold: {e['gold']}  |  Predicted: {e['predicted']}")
    print(f"Model reasoning: {e['reasoning']}")
    print()

Errors in first 30: 7

--- Error 1 ---
Headline: Panostaja Oyj 's Board also decided at its organisational meeting held upon completion of the AGM to implement the AGM decision concerning Board member fees paid as shares in such a way that shares are transferred on a quarterly basis on the date following publication of the quarterly-annual report .
Gold: neutral  |  Predicted: positive
Model reasoning: The news is about Panostaja Oyj's Board implementing a decision to transfer Board member fees in the form of shares on a quarterly basis. This suggests a long-term commitment to the company and its st

--- Error 2 ---
Headline: The Board of Directors proposes to the Shareholders ' Meeting on 18 March 2010 that the company would pay dividend for the financial year January 1 - December 31 , 2009 , EUR 0.02 per share .
Gold: neutral  |  Predicted: positive
Model reasoning: The news is positive because the company is proposing to pay a dividend, which indicates a return on investment for sha

### What we typically see

Common failure patterns on Financial PhraseBank baseline:

1. **Neutral overclaim**: model predicts neutral for anything without explicit "up/down" words. E.g., "*The company signed a partnership agreement*" gets flagged neutral, but the gold label is positive.
2. **Domain-blindness**: "*Operating profit narrowed to EUR 5.5 million from EUR 8.2 million*" often mispredicted as neutral (contains no "bad" word) but is actually negative.
3. **Directional confusion**: "*Sales fell less than expected*" is often positive in finance-speak, but the model sees "fell" and predicts negative.

These are exactly the patterns that **good examples in the prompt would fix**. But hand-picking those examples is what we want to avoid. Enter teleprompters.


### Run 1: `BootstrapFewShot`

**Algorithm** (simplified):

1. Take an initial (unoptimized) module. Call it the **teacher**.
2. For each training example $(x_i, y_i)$:
   - Run the teacher on $x_i$ to get a full trace (reasoning + output).
   - If the trace is *correct* (matches $y_i$ under our metric), keep it as a **demonstration**.
3. Take the top-$k$ correct demonstrations and inject them into the prompt for the **student**.
4. Return the student module with those baked-in demonstrations.

Why does this work? Because the demos aren't just $(x, y)$ pairs, they include the model's own reasoning trace. The model is essentially learning "here's how *I* successfully thought about similar problems before." This is called **bootstrapping** because the model helps generate its own training signal.


In [ ]:
from dspy.teleprompt import BootstrapFewShot

# Configure the teleprompter
bootstrap = BootstrapFewShot(
    metric=sentiment_match,
    max_bootstrapped_demos=4,   # how many self-generated demos to keep
    max_labeled_demos=4,        # how many raw labeled demos as fallback
    max_rounds=1,               # bootstrapping rounds
)

# Start from a fresh ChainOfThought module
student = dspy.ChainOfThought(FinancialSentiment)

print("Compiling with BootstrapFewShot... (~2 min)")
t0 = time.time()
compiled_bootstrap = bootstrap.compile(student=student, trainset=trainset)
print(f"Compile time: {time.time() - t0:.1f}s")

Compiling with BootstrapFewShot... (~2 min)


 19%|█▉        | 4/21 [00:00<00:00, 26.02it/s]

Bootstrapped 4 full traces after 4 examples for up to 1 rounds, amounting to 4 attempts.
Compile time: 0.2s


In [ ]:
# Inspect what got compiled
print("Number of demonstrations kept:", len(compiled_bootstrap.predict.demos))
print("\nFirst compiled demo:")
demo = compiled_bootstrap.predict.demos[0]


print("Headline:", demo.headline)
print("Reasoning:", demo.reasoning[:300] if hasattr(demo, "reasoning") else "n/a")
print("Sentiment:", demo.sentiment)

Number of demonstrations kept: 4

First compiled demo:
Headline: The fair value of the company 's investment properties went down to EUR 2.768 billion at the end of 2009 from EUR 2.916 billion a year earlier .
Reasoning: The decrease in the fair value of the company's investment properties suggests a potential decrease in the company's assets and possibly its overall value, which could negatively impact the stock price.
Sentiment: negative


In [ ]:
# Evaluate the compiled program
print("Evaluating BootstrapFewShot-compiled program...")
t0 = time.time()


bootstrap_score = evaluator(compiled_bootstrap)
# Access the numerical score from the EvaluationResult object
print(f"BootstrapFewShot accuracy: {bootstrap_score.score:.2f}%")
print(f"Time: {time.time() - t0:.1f}s")

Evaluating BootstrapFewShot-compiled program...
Average Metric: 28.00 / 38 (73.7%):  36%|███▋      | 37/102 [00:01<00:02, 23.92it/s]

2026/08/13 17:30:36 ERROR dspy.utils.parallelizer: Error for Example({'headline': 'According to the CEO of Nordea Bank Estonia Vahur Kraft , Nordea Finland and Stockmann have been cooperating for more than ten years .', 'sentiment': 'positive'}) (input_keys={'headline'}): [llama-3.1-8b-instant] litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01kxgmg0n2exy9y4k8azwc3b28` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 4926, Requested 1234. Please try again in 1.6s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}
. Set `provide_traceback=True` for traceback.


Average Metric: 31.00 / 41 (75.6%):  41%|████      | 42/102 [00:08<00:37,  1.62it/s]

2026/08/13 17:30:43 ERROR dspy.utils.parallelizer: Error for Example({'headline': 'For Teleste , the acquisition marks an entry into services business in a market where it has long been an established and significant supplier of products .', 'sentiment': 'positive'}) (input_keys={'headline'}): [llama-3.1-8b-instant] litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01kxgmg0n2exy9y4k8azwc3b28` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 5022, Requested 1233. Please try again in 2.55s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}
. Set `provide_traceback=True` for traceback.


Average Metric: 32.00 / 43 (74.4%):  44%|████▍     | 45/102 [00:15<01:09,  1.22s/it]

...

Average Metric: 38.00 / 49 (77.6%):  57%|█████▋    | 58/102 [00:43<01:44,  2.39s/it]

2026/08/13 17:31:18 ERROR dspy.utils.parallelizer: Error for Example({'headline': 'Qualcomm estimated a first-quarter profit between 46 and 50 cents a share , excluding certain items , below the analyst estimate of 61 cents a share .', 'sentiment': 'negative'}) (input_keys={'headline'}): [llama-3.1-8b-instant] litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01kxgmg0n2exy9y4k8azwc3b28` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 5173, Requested 828. Please try again in 10ms. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}
. Set `provide_traceback=True` for traceback.


Average Metric: 38.00 / 49 (77.6%):  58%|█████▊    | 59/102 [00:46<00:34,  1.26it/s]

2026/08/13 17:31:18 WARNING dspy.utils.parallelizer: Execution cancelled due to errors or interruption.


Exception: Execution cancelled due to errors or interruption.